In [0]:
from pyspark.sql import functions as F

PREFIX = "workspace.gep_public_procurement"

bronze = spark.table(f"{PREFIX}.bronze_precos")
catalogo = spark.table(f"{PREFIX}.ref_catalogo")
conversoes = spark.table(f"{PREFIX}.ref_conversoes")

# Um snapshot Bronze deve conter apenas uma execução.
runs = bronze.select("run_id").distinct().collect()
assert len(runs) == 1, "Esperada uma única execução na Bronze."

RUN_ID = runs[0]["run_id"]

# Primeiro lemos os campos como texto.
# As conversões e validações acontecem na etapa própria.
schema_json = """
    codigoItemCatalogo STRING,
    nomeUnidadeFornecimento STRING,
    siglaUnidadeMedida STRING,
    capacidadeUnidadeFornecimento STRING,
    quantidade STRING,
    precoUnitario STRING,
    dataResultado STRING,
    niFornecedor STRING,
    idCompraItem STRING,
    idItemCompra STRING
"""

base = (
    bronze
    .withColumn("dados", F.from_json("registro_json", schema_json))
    .select(
        "run_id",
        "codigo_solicitado",
        "extraido_em",
        "registro_json",
        "dados.*",
    )
)

ref_catalogo = catalogo.select(
    F.trim("codigoItemCatalogo").alias("codigoItemCatalogo"),
    F.upper(F.trim("nomeUnidadeFornecimento"))
        .alias("nomeUnidadeFornecimento"),
    F.upper(F.trim("unidade_canonica")).alias("unidade_canonica"),
)

ref_conversoes = conversoes.select(
    F.upper(F.trim("source_sigla")).alias("source_sigla"),
    F.upper(F.trim("canonical_unit")).alias("canonical_unit"),
    F.col("conversion_factor").cast("double").alias("conversion_factor"),
)

# Evita que chaves repetidas nas referências multipliquem registros.
assert (
    ref_catalogo.groupBy(
        "codigoItemCatalogo", "nomeUnidadeFornecimento"
    ).count().filter("count > 1").count() == 0
), "Chave duplicada no catálogo."

assert (
    ref_conversoes.groupBy("source_sigla")
    .count().filter("count > 1").count() == 0
), "Unidade duplicada na referência de conversões."

assert (
    ref_conversoes.filter(
        F.col("conversion_factor").isNull()
        | (F.col("conversion_factor") <= 0)
        | F.isnan("conversion_factor")
        | (F.abs(F.col("conversion_factor")) == float("inf"))
    ).count() == 0
), "Fator de conversão inválido."

# Contagem de entrada, saída e remoções por etapa.
auditoria = []

def registrar_etapa(ordem, etapa, df, entrada):
    saida = df.count()
    assert saida <= entrada, f"A etapa {etapa} multiplicou registros."

    auditoria.append((
        RUN_ID, ordem, etapa, entrada, saida, entrada - saida
    ))

    print(
        f"{etapa}: entrada={entrada} | "
        f"saída={saida} | removidos={entrada - saida}"
    )
    return saida

n_bronze = bronze.count()
print("Registros na Bronze:", n_bronze)

Registros na Bronze: 2000


In [0]:
# 1. Correspondência com o catálogo:
# código do registro da API + nome da unidade de fornecimento.

auditoria = []

s1 = (
    base
    .withColumn(
        "nomeUnidadeFornecimento",
        F.upper(F.trim("nomeUnidadeFornecimento")),
    )
    .join(
        ref_catalogo,
        ["codigoItemCatalogo", "nomeUnidadeFornecimento"],
        "inner",
    )
)

n1 = registrar_etapa(1, "correspondencia_catalogo", s1, n_bronze)


# 2. Tipos e valores válidos
tipados = (
    s1
    .withColumn(
        "capacidade_original",
        F.col("capacidadeUnidadeFornecimento"),
    )
    .withColumn(
        "quantidade",
        F.expr("try_cast(quantidade AS DOUBLE)"),
    )
    .withColumn(
        "precoUnitario",
        F.expr("try_cast(precoUnitario AS DOUBLE)"),
    )
    .withColumn(
        "capacidadeUnidadeFornecimento",
        F.expr("try_cast(capacidadeUnidadeFornecimento AS DOUBLE)"),
    )
    .withColumn(
        "dataResultado",
        F.expr("try_cast(dataResultado AS DATE)"),
    )
    .withColumn(
        "niFornecedor",
        F.when(
            F.length(F.trim("niFornecedor")) > 0,
            F.trim("niFornecedor"),
        ),
    )
)

def numero_finito(nome):
    coluna = F.col(nome)
    return (
        coluna.isNotNull()
        & ~F.isnan(coluna)
        & (F.abs(coluna) != F.lit(float("inf")))
    )

# Capacidade ausente será tratada como 1 na conversão.
# Capacidade informada, mas não numérica ou infinita, é inválida.
capacidade_valida = (
    F.col("capacidade_original").isNull()
    | numero_finito("capacidadeUnidadeFornecimento")
)

s2 = tipados.filter(
    numero_finito("quantidade")
    & (F.col("quantidade") > 0)
    & numero_finito("precoUnitario")
    & (F.col("precoUnitario") > 0)
    & F.col("dataResultado").isNotNull()
    & capacidade_valida
)

n2 = registrar_etapa(2, "tipos_valores_validos", s2, n1)


# 3. Duplicidades exatas do registro recebido.
# Metadados da extração não participam da comparação.
s3 = s2.dropDuplicates(["registro_json"])

n3 = registrar_etapa(3, "duplicidades_exatas", s3, n2)


# 4. Unidades reconhecidas e compatíveis com o catálogo
s4 = (
    s3
    .withColumn(
        "siglaUnidadeMedida",
        F.upper(F.trim("siglaUnidadeMedida")),
    )
    .join(
        ref_conversoes,
        F.col("siglaUnidadeMedida") == F.col("source_sigla"),
        "inner",
    )
    .filter(F.col("canonical_unit") == F.col("unidade_canonica"))
)

n4 = registrar_etapa(4, "unidades_compativeis", s4, n3)


# 5. Conversão para a unidade canônica
def converter_unidades(df):
    return (
        df
        .withColumn(
            "effective_capacity",
            F.when(
                F.col("capacidadeUnidadeFornecimento") > 0,
                F.col("capacidadeUnidadeFornecimento")
            ).otherwise(F.lit(1.0))
        )
        .withColumn(
            "canonical_quantity",
            F.col("quantidade")
            * F.col("effective_capacity")
            * F.col("conversion_factor")
        )
        .withColumn(
            "canonical_price",
            F.col("precoUnitario")
            / (
                F.col("effective_capacity")
                * F.col("conversion_factor")
            )
        )
    )


s5 = converter_unidades(s4)

n5 = registrar_etapa(5, "conversao_canonica", s5, n4)

assert n5 == n4, "A conversão alterou a quantidade de registros."

# Conversão não deve descartar linhas nem produzir valores inválidos.
assert n5 == n4, "A conversão alterou a quantidade de registros."

conversao_valida = (
    numero_finito("canonical_quantity")
    & (F.col("canonical_quantity") > 0)
    & numero_finito("canonical_price")
    & (F.col("canonical_price") > 0)
)

assert (
    s5.filter(~F.coalesce(conversao_valida, F.lit(False))).count() == 0
), "A conversão produziu valores nulos, não positivos ou não finitos."

print("\nTransformações da Silver concluídas.")

correspondencia_catalogo: entrada=2000 | saída=1461 | removidos=539
tipos_valores_validos: entrada=1461 | saída=1461 | removidos=0
duplicidades_exatas: entrada=1461 | saída=1461 | removidos=0
unidades_compativeis: entrada=1461 | saída=1439 | removidos=22
conversao_canonica: entrada=1439 | saída=1439 | removidos=0

Transformações da Silver concluídas.


In [0]:
silver = s5.select(
    "run_id",
    "codigo_solicitado",
    "extraido_em",
    "codigoItemCatalogo",
    "nomeUnidadeFornecimento",
    "idCompraItem",
    "idItemCompra",
    "niFornecedor",
    "dataResultado",
    "siglaUnidadeMedida",
    "unidade_canonica",
    "quantidade",
    "precoUnitario",
    "capacidadeUnidadeFornecimento",
    "effective_capacity",
    "conversion_factor",
    "canonical_quantity",
    "canonical_price",
    "registro_json",
)

auditoria_df = spark.createDataFrame(
    auditoria,
    schema="""
        run_id STRING,
        ordem INT,
        etapa STRING,
        linhas_entrada LONG,
        linhas_saida LONG,
        linhas_removidas LONG
    """,
)

(
    auditoria_df.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.auditoria_silver")
)

(
    silver.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.silver_precos")
)

assert (
    n_bronze - sum(linha[5] for linha in auditoria) == n5
), "As contagens de remoção não reconciliam com a Bronze."

print("Tabela salva:", f"{PREFIX}.silver_precos")
print("Registros na Silver:", n5)

if n5 == 0:
    print(
        "ATENÇÃO: nenhuma linha passou pelas regras. "
        "Investigar antes de seguir para a Gold."
    )

display(auditoria_df.orderBy("ordem"))

display(
    silver.select(
        "codigoItemCatalogo",
        "nomeUnidadeFornecimento",
        "dataResultado",
        "quantidade",
        "precoUnitario",
        "effective_capacity",
        "unidade_canonica",
        "canonical_quantity",
        "canonical_price",
    ).limit(10)
)

Tabela salva: workspace.gep_public_procurement.silver_precos
Registros na Silver: 1439


run_id,ordem,etapa,linhas_entrada,linhas_saida,linhas_removidas
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,1,correspondencia_catalogo,2000,1461,539
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,2,tipos_valores_validos,1461,1461,0
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,3,duplicidades_exatas,1461,1461,0
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,4,unidades_compativeis,1461,1439,22
a7ff4fb4-9ed6-485d-a4d0-3611b361f67b,5,conversao_canonica,1439,1439,0


codigoItemCatalogo,nomeUnidadeFornecimento,dataResultado,quantidade,precoUnitario,effective_capacity,unidade_canonica,canonical_quantity,canonical_price
217129,PACOTE,2026-06-01,150.0,5.05,400.0,KG,60.0,12.624999999999998
226093,PACOTE,2026-05-29,210.0,11.8,100.0,UN,21000.0,0.11800000000000001
226093,PACOTE,2026-05-15,20.0,19.69,100.0,UN,2000.0,0.19690000000000002
226093,PACOTE,2026-05-06,500.0,19.15,100.0,UN,50000.0,0.19149999999999998
226093,PACOTE,2026-05-04,5000.0,1.29,10.0,UN,50000.0,0.129
226093,PACOTE,2026-04-13,14000.0,9.5,100.0,UN,1400000.0,0.095
226093,PACOTE,2026-04-01,40.0,12.63,100.0,UN,4000.0,0.1263
226094,PACOTE,2025-12-04,732.0,10.37,5.0,UN,3660.0,2.074
226094,PACOTE,2025-06-30,100.0,3.52,100.0,UN,10000.0,0.0352
237589,CAIXA,2026-11-05,400.0,4.15,30.0,UN,12000.0,0.13833333333333334


In [0]:
import math

# Dados fictícios exclusivamente para teste.
# Resultados esperados calculados manualmente.
casos = spark.createDataFrame(
    [
        ("gramas_para_kg",    10.0, 10.0, 500.0, "G",  5.0, 20.0),
        ("kg_sem_conversao",   3.0, 50.0,   5.0, "KG", 15.0, 10.0),
        ("capacidade_zero",    4.0,  8.0,   0.0, "UN", 4.0,  8.0),
        ("capacidade_nula",    2.0,  6.0,  None, "UN", 2.0,  6.0),
        ("capacidade_negativa",3.0,  9.0,  -2.0, "UN", 3.0,  9.0),
    ],
    """
    caso STRING,
    quantidade DOUBLE,
    precoUnitario DOUBLE,
    capacidadeUnidadeFornecimento DOUBLE,
    source_sigla STRING,
    quantidade_esperada DOUBLE,
    preco_esperado DOUBLE
    """
)

# Usa os fatores da própria tabela de referência do projeto.
casos_com_fator = casos.join(
    ref_conversoes.select("source_sigla", "conversion_factor"),
    on="source_sigla",
    how="left"
)

assert casos_com_fator.count() == 5
assert (
    casos_com_fator
    .filter(F.col("conversion_factor").isNull())
    .count() == 0
), "Falta fator de conversão para um dos testes."

calculados = converter_unidades(casos_com_fator)

resultados_teste = []

for linha in calculados.collect():
    quantidade_ok = math.isclose(
        linha["canonical_quantity"],
        linha["quantidade_esperada"],
        rel_tol=1e-9,
        abs_tol=1e-9
    )

    preco_ok = math.isclose(
        linha["canonical_price"],
        linha["preco_esperado"],
        rel_tol=1e-9,
        abs_tol=1e-9
    )

    resultados_teste.append((
        linha["caso"],
        linha["canonical_quantity"],
        linha["quantidade_esperada"],
        linha["canonical_price"],
        linha["preco_esperado"],
        "OK" if quantidade_ok and preco_ok else "FALHOU"
    ))

evidencia_testes = spark.createDataFrame(
    resultados_teste,
    """
    caso STRING,
    quantidade_calculada DOUBLE,
    quantidade_esperada DOUBLE,
    preco_calculado DOUBLE,
    preco_esperado DOUBLE,
    resultado STRING
    """
)

(
    evidencia_testes.write
    .format("delta")
    .mode("overwrite")
    .saveAsTable(f"{PREFIX}.testes_conversao")
)

display(evidencia_testes)

assert all(
    resultado[-1] == "OK"
    for resultado in resultados_teste
), "Um ou mais testes de conversão falharam."

print("Os 5 testes de conversão passaram.")

caso,quantidade_calculada,quantidade_esperada,preco_calculado,preco_esperado,resultado
gramas_para_kg,5.0,5.0,20.0,20.0,OK
kg_sem_conversao,15.0,15.0,10.0,10.0,OK
capacidade_zero,4.0,4.0,8.0,8.0,OK
capacidade_nula,2.0,2.0,6.0,6.0,OK
capacidade_negativa,3.0,3.0,9.0,9.0,OK


Os 5 testes de conversão passaram.
